# 05 - Comparison methods B1-B4

**Settings:** Accelerator = **None (CPU)**, Internet = On. Run with **Save Version > Save & Run All (Commit)**,
not the editor's Run All: a committed run keeps going when the laptop is closed and its output is saved.

**Inputs** (Add Input > Your Work > Notebooks):
1. `01_prepare_trainval` (ufm-trainval)
2. `02_prepare_test` (ufm-test)

**Output:** `runs/baselines/`  |  **Approx. time:** 1.5-3 h (no GPU quota used)

B1 intensity change (Otsu), B2 coherence threshold 0.3, B3 rule-based tree, B4 random forest with validation-tuned class weights; all on 5x5 median-filtered SAR.

In [ ]:
# ---- Settings -------------------------------------------------------------
GITHUB_REPO = "https://github.com/mhdnasim6-boop/urban-flood-mapping.git"
BRANCH = "main"
# ---------------------------------------------------------------------------
# Private repo: add a Kaggle secret named GITHUB_TOKEN (Add-ons > Secrets).
# Optional: a secret named HF_TOKEN (Hugging Face read token) avoids download rate limits.
import glob, os, subprocess
from pathlib import Path

def secret(name):
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret(name)
    except Exception:
        return None

url = GITHUB_REPO
if secret("GITHUB_TOKEN"):
    url = url.replace("https://", f"https://{secret('GITHUB_TOKEN')}@")
if secret("HF_TOKEN"):
    os.environ["HF_TOKEN"] = secret("HF_TOKEN")
subprocess.run(["rm", "-rf", "/tmp/ufm"])
subprocess.run(["git", "clone", "-q", "--depth", "1", "-b", BRANCH, url, "/tmp/ufm"], check=True)
os.chdir("/tmp/ufm")
print("code version:", subprocess.run(["git", "log", "-1", "--oneline"], capture_output=True, text=True).stdout)

def find_input(name, notebook):
    """Locate an attached input folder or stop immediately with a clear message."""
    hits = sorted(glob.glob(f"/kaggle/input/**/{name}", recursive=True))
    if not hits:
        raise SystemExit(f"STOP: input '{name}' not found. Add the output of notebook {notebook} "
                         "(Add Input > Your Work > Notebooks) and run again.")
    print(f"{name}: {hits[0]}")
    return hits[0]

In [ ]:
TRAINVAL = find_input("ufm-trainval", "01_prepare_trainval")
TEST = find_input("ufm-test", "02_prepare_test")

In [ ]:
!pip install -q rasterio tabulate 2>/dev/null; python -c "import cv2, xgboost; print('cv2', cv2.__version__, 'xgboost', xgboost.__version__)"

In [ ]:
!python scripts/run_baselines.py --test-root {TEST} --trainval-root {TRAINVAL} --out /kaggle/working/runs/baselines --rf-device cpu
!cat /kaggle/working/runs/baselines/settings.json